In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
from scipy.integrate import quad
from scipy.special import gamma
from scipy.stats import norm

# ==============================================================================
# 1. 數理統計常數計算核心
# ==============================================================================
d3_table = {
    2: 0.8525, 3: 0.8884, 4: 0.8798, 5: 0.8641, 6: 0.8480,
    7: 0.8332, 8: 0.8198, 9: 0.8078, 10: 0.7971, 11: 0.7873,
    12: 0.7785, 13: 0.7704, 14: 0.7630, 15: 0.7562, 16: 0.7499,
    17: 0.7441, 18: 0.7386, 19: 0.7335, 20: 0.7287, 21: 0.7242,
    22: 0.7199, 23: 0.7159, 24: 0.7121, 25: 0.7084
}

range_factor_table = {
    2: (0.000, 3.686, 0.000, 3.267),
    3: (0.000, 4.358, 0.000, 2.574),
    4: (0.000, 4.698, 0.000, 2.282),
    5: (0.000, 4.918, 0.000, 2.114),
    6: (0.000, 5.078, 0.000, 2.004),
    7: (0.204, 5.204, 0.076, 1.924),
    8: (0.388, 5.306, 0.136, 1.864),
    9: (0.547, 5.393, 0.184, 1.816),
    10: (0.687, 5.469, 0.223, 1.777),
    11: (0.811, 5.535, 0.256, 1.744),
    12: (0.922, 5.594, 0.283, 1.717),
    13: (1.025, 5.647, 0.307, 1.693),
    14: (1.118, 5.696, 0.328, 1.672),
    15: (1.203, 5.741, 0.347, 1.653),
    16: (1.282, 5.782, 0.363, 1.637),
    17: (1.356, 5.820, 0.378, 1.622),
    18: (1.424, 5.856, 0.391, 1.608),
    19: (1.487, 5.891, 0.403, 1.597),
    20: (1.549, 5.921, 0.415, 1.585),
    21: (1.605, 5.951, 0.425, 1.575),
    22: (1.659, 5.979, 0.434, 1.566),
    23: (1.710, 6.006, 0.443, 1.557),
    24: (1.759, 6.031, 0.451, 1.548),
    25: (1.806, 6.056, 0.459, 1.541),
}


def calculate_factors(n: int):
    """計算指定樣本數 n 的所有 SPC 係數"""
    if n is None or not 2 <= int(n) <= 25:
        raise ValueError("樣本數 n 必須介於 2 到 25。")
    n = int(n)

    # 1. 計算 c4
    c4 = np.sqrt(2.0 / (n - 1)) * (gamma(n / 2.0) / gamma((n - 1) / 2.0))

    # 2. 數值積分計算 d2
    integrand_d2 = lambda x: 1.0 - (1.0 - norm.cdf(x))**n - (norm.cdf(x))**n
    d2, _ = quad(integrand_d2, -10, 10)

    # d3 是全距標準差係數，使用標準 SPC 常數表。
    d3 = d3_table[n]

    # 3. 計算衍生係數
    A = 3.0 / np.sqrt(n)
    A2 = 3.0 / (d2 * np.sqrt(n))
    A3 = 3.0 / (c4 * np.sqrt(n))
    c5 = np.sqrt(max(0.0, 1.0 - c4**2))
    B3 = max(0.0, 1.0 - 3.0 * (c5 / c4))
    B4 = 1.0 + 3.0 * (c5 / c4)
    B5 = max(0.0, c4 - 3.0 * c5)
    B6 = c4 + 3.0 * c5
    # 使用標準常數表的精度，避免從已四捨五入的 d2/d3 推導時產生差異。
    D1, D2, D3, D4 = range_factor_table[n]

    return {
        "n": n,
        "A": round(A, 3),
        "A2": round(A2, 3),
        "A3": round(A3, 3),
        "c4": round(c4, 4),
        "1/c4": round(1.0 / c4, 4),
        "B3": round(B3, 3),
        "B4": round(B4, 3),
        "B5": round(B5, 3),
        "B6": round(B6, 3),
        "d2": round(d2, 3),
        "1/d2": round(1.0 / d2, 4),
        "d3": round(d3, 3),
        "D1": round(D1, 3),
        "D2": round(D2, 3),
        "D3": round(D3, 3),
        "D4": round(D4, 3),
    }


def get_single_n_dataframe(n: int):
    """點擊按鈕時觸發：將指定樣本數 n 的計算結果封裝成結構化表格"""
    if n is None:
        n = 5
    f = calculate_factors(n)
    return pd.DataFrame([f]).rename(columns={
        "n": "樣本數 (n)",
        "A": "A (X-bar)",
        "A2": "A2 (X-bar-R)",
        "A3": "A3 (X-bar-s)",
        "c4": "c4",
        "1/c4": "1/c4",
        "B3": "B3",
        "B4": "B4",
        "B5": "B5",
        "B6": "B6",
        "d2": "d2",
        "1/d2": "1/d2",
        "d3": "d3",
        "D1": "D1",
        "D2": "D2",
        "D3": "D3",
        "D4": "D4",
    })


def get_full_table():
    """產生 n=2 到 25 的完整常數對照表 DataFrame"""
    rows = [calculate_factors(i) for i in range(2, 26)]
    df = pd.DataFrame(rows)
    df.rename(columns={
        "n": "樣本數 (n)",
        "A": "A",
        "A2": "A2",
        "A3": "A3",
        "c4": "c4",
        "1/c4": "1/c4",
        "B3": "B3",
        "B4": "B4",
        "B5": "B5",
        "B6": "B6",
        "d2": "d2",
        "1/d2": "1/d2",
        "d3": "d3",
        "D1": "D1",
        "D2": "D2",
        "D3": "D3",
        "D4": "D4",
    }, inplace=True)
    return df[
        [
            "樣本數 (n)", "A", "A2", "A3", "c4", "1/c4",
            "B3", "B4", "B5", "B6", "d2", "1/d2",
            "d3", "D1", "D2", "D3", "D4",
        ]
    ]

# 預先載入完整對照表
DF_FULL = get_full_table()

# ==============================================================================
# 2. 建立 Gradio Web 介面
# ==============================================================================
with gr.Blocks(title="SPC 管制圖常數計算系統") as demo:
    gr.Markdown("# 📊 SPC 管制圖常數計算與速查系統")
    gr.Markdown("請在下方輸入子組樣本數 $n$（純數字輸入），**點擊「⚡ 計算參數」按鈕後**系統將以表格呈現計算結果。")
    
    # --- 上方：純數字輸入區與觸發按鈕 ---
    with gr.Row():
        with gr.Column(scale=3):
            input_n = gr.Number(
                value=5, 
                label="👉 請輸入子組樣本數 n (正整數，例如: 5)", 
                precision=0,
                minimum=2,
                maximum=25
            )
        with gr.Column(scale=1):
            btn_calc = gr.Button("⚡ 計算參數", variant="primary")
            
    # --- 中間：單一樣本數的詳細參數表格 ---
    gr.Markdown("### 📌 計算結果參數表")
    table_single = gr.DataFrame(
        value=get_single_n_dataframe(5),
        interactive=False,
        wrap=True
    )
    
    gr.Markdown("---")
    
    # --- 下方：完整常數對照表 ---
    gr.Markdown(
        "### 📋 完整管制圖常數對照表 (n = 2 ~ 25)\n"
        "包含平均數圖、標準差圖及全距圖的管制界限係數與中心線係數。"
    )
    table_full = gr.DataFrame(
        value=DF_FULL, 
        interactive=False, 
        wrap=True
    )

    # 【重要修正】：只在點擊「計算參數」按鈕時才執行計算
    btn_calc.click(
        fn=get_single_n_dataframe, 
        inputs=[input_n], 
        outputs=[table_single]
    )
    # 同步支援在輸入框按下 Enter 鍵送出計算
    input_n.submit(
        fn=get_single_n_dataframe, 
        inputs=[input_n], 
        outputs=[table_single]
    )

# 啟動介面
if __name__ == "__main__":
    demo.launch()